In [1]:
from peft import PeftModel
from unsloth import FastLanguageModel
from transformers import AutoModelForCausalLM
import torch
from tqdm import tqdm
from task1.test_ import CLPsychDataLoader, create_instruction_dataset, df_to_training_format
from task1.test_ import predict_abcd, parse_json_output



# Weighted forward pass
def ensemble_forward(input_ids, weights=[0.4, 0.35, 0.25]):  # Tune weights
    with torch.no_grad():
        out_l = model_l(input_ids=input_ids).logits
        out_q = model_q(input_ids=input_ids).logits  
        out_p = model_p(input_ids=input_ids).logits
        ensemble_logits = (weights[0] * out_l + weights[1] * out_q + weights[2] * out_p)
    return ensemble_logits.argmax(-1)


# ========== STEP 1: Load Data ==========
print("=" * 60)
print("STEP 1: Loading Data")
print("=" * 60)
test_loader = CLPsychDataLoader('tasks12/', split='test')
test_df = test_loader.load()
print("\n" + "=" * 60)
print("Test Set Stats")
test_loader.verify_order()
# ========== STEP 2: Create Instruction Dataset ==========
print("\n" + "=" * 60)
print("STEP 2: Creating Instruction Dataset")
print("=" * 60)

test_df = create_instruction_dataset(test_df)

test_data = df_to_training_format(test_df[:1])




/home/snath/miniconda3/envs/lus/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_2757088/4212715715.py:2: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
STEP 1: Loading Data
Loaded 7256d10881 with 11 posts.
Loaded 49b6f3373c with 8 posts.
Loaded c2e20efd4f with 10 posts.
Loaded f55ec24987 with 11 posts.
Loaded 1aaf23187c with 10 posts.
Loaded 78125e9967 with 10 posts.
Loaded 0b14f9798b with 10 posts.
Loaded 0a78977ba9 with 5 posts.
Loaded 207f2fac5e with 11 posts.
Loaded 7a6b02d34a with 6 posts.
Loaded 92 posts from 10 timelines

Test Set Stats

=== Verifying Post Order ===
✅ All posts are in correct order

STEP 2: Creating Instruction Dataset

Created 92 instruction examples
From 10 timelines


In [2]:
# ========== 1. Load Trained Model ==========
print("="*60)
print("Loading trained model...")
print("="*60)
# base_l = AutoModelForCausalLM.from_pretrained("unsloth/llama-3-8b-Instruct-bnb-4bit")  # Shared base if possible
# base_q = AutoModelForCausalLM.from_pretrained("unsloth/Qwen3.5-9B-GGUF")
# base_p = AutoModelForCausalLM.from_pretrained("unsloth/phi-4-bnb-4bit")

# Load all LoRAs
model_l, tokenizer_l = FastLanguageModel.from_pretrained(
        model_name="llama-final",  # Your checkpoint directory
        max_seq_length=2048,
        dtype=None,
        device_map="auto",
        load_in_4bit=True,
    )

FastLanguageModel.for_inference(model_l)
print("✅ Llama Model loaded\n")
# model_q, tokenizer_q = FastLanguageModel.from_pretrained(
#         model_name="qwen-final",  # Your checkpoint directory
#         max_seq_length=2048,
#         dtype=None,
#         device_map="auto",
#         load_in_4bit=True,
#     )
# FastLanguageModel.for_inference(model_q)
# print("✅ QWEN Model loaded\n")
model_p, tokenizer_p = FastLanguageModel.from_pretrained(
        model_name="phi-final",  # Your checkpoint directory
        max_seq_length=2048,
        dtype=None,
        device_map="auto",
        load_in_4bit=True,
    )
FastLanguageModel.for_inference(model_p)
print("✅ Phi Model loaded\n")



Loading trained model...
==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.558 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 291/291 [00:01<00:00, 148.07it/s, Materializing param=model.norm.weight]                              
Unsloth: Will load unsloth/llama-3-8b-Instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.3.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


✅ Llama Model loaded

==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.558 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 363/363 [00:02<00:00, 126.84it/s, Materializing param=model.norm.weight]                              
Unsloth 2026.3.4 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


✅ Phi Model loaded



In [4]:
# Generate predictions
predictions = []

for item in tqdm(test_data, desc="Predicting"):
    # print(item)
    try:
    # Generate
        response_l = predict_abcd(
            item['instruction'],
            item['input'],
            model_l, 
            tokenizer_l
        )

        response_q = predict_abcd(
            item['instruction'],
            item['input'],
            model_p, 
            tokenizer_p
        )
        print(f"\nLlama Prediction: {response_l}")
        print(f"Phi Prediction: {response_q}")
        # Parse
        # prediction = parse_json_output(response_l)
        
        # predictions.append({
        #     'timeline_id':item['timeline_id'],
        #     'post_id': item['post_id'],
        #     'prediction': prediction
        # })
        
    except Exception as e:
        print(f"\n❌ Error: {e}")
        predictions.append({
            'prediction': None,
            'ground_truth': None,
            'error': str(e)
        })

print(f"\n✅ Generated {len(predictions)} predictions\n")

Predicting:   0%|          | 0/1 [00:00<?, ?it/s]

{'input_ids': tensor([[128000, 128000, 128006,  ...,  78191, 128007,    271]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1]], device='cuda:0')}
torch.Size([1, 1098, 128256])
{'input_ids': tensor([[100264,   9125, 100266,  ..., 100264,  78191, 100266]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1]], device='cuda:0')}


Predicting: 100%|██████████| 1/1 [00:00<00:00,  1.80it/s]


❌ Error: CUDA out of memory. Tried to allocate 38.00 MiB. GPU 0 has a total capacity of 23.56 GiB of which 18.62 MiB is free. Including non-PyTorch memory, this process has 23.23 GiB memory in use. Of the allocated memory 22.75 GiB is allocated by PyTorch, and 147.82 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

✅ Generated 1 predictions

